### Allscripts Sunrise (SCM) Drug Exposure Investigation

This notebook isolates why `_exponent.omop_scm.drug_exposure.drug_concept_id` is all zero.
Run this on the branch version of `scm-failures` and share the outputs so we can decide whether the fix is quick enough to unblock `drug_era` and `dose_era` today.

In [ ]:
%sql
SELECT COUNT(*) AS scm_drug_exposure_rows
FROM _exponent.omop_scm.drug_exposure;

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) AS drug_concept_id_zero_rows,
  SUM(CASE WHEN drug_source_concept_id = 0 THEN 1 ELSE 0 END) AS drug_source_concept_id_zero_rows,
  SUM(CASE WHEN drug_source_value IS NULL THEN 1 ELSE 0 END) AS null_drug_source_value_rows,
  SUM(CASE WHEN person_id IS NULL THEN 1 ELSE 0 END) AS null_person_rows,
  SUM(CASE WHEN drug_exposure_start_date IS NULL THEN 1 ELSE 0 END) AS null_start_date_rows
FROM _exponent.omop_scm.drug_exposure;

In [ ]:
%sql
SELECT
  drug_source_value,
  route_source_value,
  dose_unit_source_value,
  quantity,
  days_supply,
  COUNT(*) AS row_count
FROM _exponent.omop_scm.drug_exposure
GROUP BY drug_source_value, route_source_value, dose_unit_source_value, quantity, days_supply
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  drug_source_value,
  COUNT(*) AS row_count
FROM _exponent.omop_scm.drug_exposure
WHERE drug_source_value IS NOT NULL
GROUP BY drug_source_value
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  d.drug_source_value,
  MAX(dstc.omop_concept_id) AS mapped_omop_concept_id,
  COUNT(*) AS row_count
FROM _exponent.omop_scm.drug_exposure d
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept dstc
  ON UPPER(TRIM(COALESCE(dstc.source_value, dstc.source_id))) = UPPER(TRIM(d.drug_source_value))
 AND dstc.domain_id = 'Drug'
 AND dstc.source_system = 'allscripts_scm'
 AND dstc.active_flag = TRUE
WHERE d.drug_source_value IS NOT NULL
GROUP BY d.drug_source_value
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  COUNT(DISTINCT drug_source_value) AS distinct_drug_source_values,
  COUNT(DISTINCT CASE WHEN drug_source_value IS NOT NULL THEN UPPER(TRIM(drug_source_value)) END) AS distinct_normalized_drug_source_values,
  COUNT(DISTINCT CASE WHEN dstc.omop_concept_id IS NOT NULL AND dstc.omop_concept_id <> 0 THEN UPPER(TRIM(d.drug_source_value)) END) AS normalized_values_with_nonzero_mapping
FROM _exponent.omop_scm.drug_exposure d
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept dstc
  ON UPPER(TRIM(COALESCE(dstc.source_value, dstc.source_id))) = UPPER(TRIM(d.drug_source_value))
 AND dstc.domain_id = 'Drug'
 AND dstc.source_system = 'allscripts_scm'
 AND dstc.active_flag = TRUE;

In [ ]:
%sql
SELECT
  source_id,
  source_value,
  omop_concept_id,
  source_code_description,
  COUNT(*) OVER () AS total_mapping_rows
FROM _exponent.omop_mapping.domain_source_to_concept
WHERE domain_id = 'Drug'
  AND source_system = 'allscripts_scm'
  AND active_flag = TRUE
LIMIT 100;

In [ ]:
%sql
SELECT
  ord.Name,
  ord.IDCode,
  ord.TypeCode,
  ord.CareProviderGUID,
  medext.DispenseAmount,
  medext.DispenseAmountUnit,
  medext.RouteCode,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.OrderGUID = ord.GUID
WHERE ord.Active = TRUE
GROUP BY ord.Name, ord.IDCode, ord.TypeCode, ord.CareProviderGUID, medext.DispenseAmount, medext.DispenseAmountUnit, medext.RouteCode
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  ord.Name,
  ord.IDCode,
  ord.TypeCode,
  COUNT(*) AS row_count,
  MAX(drug_concept.omop_concept_id) AS mapped_drug_concept_id,
  MAX(route_concept.omop_concept_id) AS mapped_route_concept_id
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept drug_concept
  ON drug_concept.domain_id = 'Drug'
 AND drug_concept.source_system = 'allscripts_scm'
 AND drug_concept.active_flag = TRUE
 AND UPPER(TRIM(COALESCE(drug_concept.source_value, drug_concept.source_id))) IN (
      UPPER(TRIM(ord.Name)),
      UPPER(TRIM(ord.IDCode))
 )
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.OrderGUID = ord.GUID
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.domain_id = 'Route'
 AND route_concept.source_system = 'allscripts_scm'
 AND route_concept.active_flag = TRUE
 AND UPPER(TRIM(COALESCE(route_concept.source_value, route_concept.source_id))) = UPPER(TRIM(medext.RouteCode))
WHERE ord.Active = TRUE
GROUP BY ord.Name, ord.IDCode, ord.TypeCode
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT *
FROM _exponent.omop_scm.drug_exposure
LIMIT 100;